In [ ]:
import os
import pandas as pd

from mlxtend.frequent_patterns import (
    apriori,
    association_rules
)

from mlxtend.preprocessing import TransactionEncoder


class MarketBasketAnalysis:

    def __init__(self, filepath):

        self.filepath = filepath

        os.makedirs("../outputs/recommendation", exist_ok=True)

    ##########################################################

    def load_data(self):

        print("="*60)
        print("Loading Sales Dataset")
        print("="*60)

        self.df = pd.read_csv(

            self.filepath,

            dtype={
                "Invoice":str,
                "StockCode":str
            },

            low_memory=False

        )

        print(self.df.shape)

    ##########################################################

    def prepare_transactions(self):

        print("\nPreparing Transactions...\n")

        transactions=(

            self.df

            .groupby("Invoice")["Description"]

            .apply(list)

            .tolist()

        )

        self.transactions=transactions

        print("Transactions :",len(transactions))

    ##########################################################

    def build_apriori(self):

        print("\nRunning Apriori...\n")

        te=TransactionEncoder()

        te_array=te.fit(

            self.transactions

        ).transform(

            self.transactions

        )

        basket=pd.DataFrame(

            te_array,

            columns=te.columns_

        )

        frequent=apriori(

            basket,

            min_support=0.02,

            use_colnames=True

        )

        rules=association_rules(

            frequent,

            metric="lift",

            min_threshold=1

        )

        self.frequent=frequent

        self.rules=rules

        print()

        print("Frequent Itemsets :",len(frequent))

        print("Rules :",len(rules))

    ##########################################################

    def save(self):

        self.frequent.to_csv(

            "../outputs/recommendation/frequent_itemsets.csv",

            index=False

        )

        self.rules.to_csv(

            "../outputs/recommendation/association_rules.csv",

            index=False

        )

        print("\nRecommendation Files Saved")

    ##########################################################

    def run(self):

        self.load_data()

        self.prepare_transactions()

        self.build_apriori()

        self.save()

        print("\nRecommendation Engine Completed")


if __name__=="__main__":

    obj=MarketBasketAnalysis(

        r"C:\Users\ka843\Coding\Amdox Internship_project\data\clean\clean_sales.csv"

    )

    obj.run()